# Interactive Point Cloud and BRep Visualization Explorer

This notebook provides a suite of tools for the interactive analysis of point cloud processing results generated by the `integrated_pipeline.py` script. You can use it to:

1.  **Load and select** different processing outputs from the `checkpoints` directory.
2.  **Visualize** the original and segmented point clouds in an interactive 3D viewer.
3.  **Render** the extracted BRep primitive models (planes, spheres, cylinders) overlaid on the point clouds.
4.  **Control** the visualization with interactive widgets to toggle components, filter segments, and adjust opacity.
5.  **Analyze** segmentation quality with statistical plots.
6.  **Generate** an automated summary report of the findings.

In [ ]:
import os
import json
import glob
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Layout
import matplotlib.pyplot as plt

# --- Configuration ---
CHECKPOINTS_DIR = Path('./checkpoints')

# --- Initial Setup ---
plt.style.use('seaborn-v0_8-talk')
print(f"Open3D version: {o3d.__version__}")
print(f"Plotly version: {go.__version__}")
print(f"ipywidgets version: {widgets.__version__}")

## 1. Load Processed Data

First, we need to find the available processing results in the `checkpoints` directory. The dropdown menu below will be populated with the base names of the files found. Select a file to begin the analysis.

In [ ]:
def find_processed_files(directory):
    """Finds pairs of segmented clouds and BRep JSON files."""
    segmented_files = glob.glob(f"{directory}/*_segmented.ply")
    brep_files = glob.glob(f"{directory}/*_brep.json")
    
    file_map = {}
    for seg_file in segmented_files:
        base_name = Path(seg_file).stem.replace('_segmented', '')
        json_path = directory / f"{base_name}_brep.json"
        if str(json_path) in brep_files:
            file_map[base_name] = {
                "segmented_cloud": seg_file,
                "brep_json": str(json_path)
            }
    return file_map

file_map = find_processed_files(CHECKPOINTS_DIR)

if not file_map:
    print(f"\033[91mError: No processed files found in '{CHECKPOINTS_DIR}'.\033[0m")
    print("Please run 'integrated_pipeline.py' first to generate results.")
else:
    file_selector = widgets.Dropdown(
        options=list(file_map.keys()),
        description='Select File:',
        style={'description_width': 'initial'}
    )
    display(file_selector)

## 2. BRep Primitive Geometry Generation

These helper functions are used to create 3D mesh objects from the BRep parameters stored in the JSON files. This allows us to visualize the fitted primitives in Plotly.

In [ ]:
def create_plane_mesh(equation, center, size=1.0):
    """Creates a rectangular mesh from a plane equation."""
    a, b, c, d = equation
    normal = np.array([a, b, c])
    normal = normal / np.linalg.norm(normal)
    
    # Create two orthogonal vectors in the plane
    u = np.cross(normal, [0, 0, 1]) if np.linalg.norm(np.cross(normal, [0, 0, 1])) > 0.1 else np.cross(normal, [1, 0, 0])
    u = u / np.linalg.norm(u)
    v = np.cross(normal, u)
    
    # Define the 4 corners of the plane
    corners = np.array([
        center + size * u + size * v,
        center - size * u + size * v,
        center - size * u - size * v,
        center + size * u - size * v
    ])
    
    return go.Mesh3d(x=corners[:,0], y=corners[:,1], z=corners[:,2], i=[0,0], j=[1,2], k=[2,3], opacity=0.5, color='cyan')

def create_sphere_mesh(center, radius, resolution=20):
    """Creates a sphere mesh."""
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones(np.size(u)), np.cos(v))
    return go.Surface(x=x, y=y, z=z, opacity=0.5, colorscale=[[0, 'magenta'], [1, 'magenta']], showscale=False)

def create_cylinder_mesh(center, radius, height, resolution=50):
    """Creates a cylinder mesh oriented along the z-axis."""
    z = np.linspace(center[2] - height/2, center[2] + height/2, 2)
    theta = np.linspace(0, 2*np.pi, resolution)
    theta, z = np.meshgrid(theta, z)
    x = center[0] + radius * np.cos(theta)
    y = center[1] + radius * np.sin(theta)
    return go.Surface(x=x, y=y, z=z, opacity=0.5, colorscale=[[0, 'lightgreen'], [1, 'lightgreen']], showscale=False)

def generate_brep_meshes(brep_data):
    """Generates a list of Plotly meshes from BRep data."""
    meshes = []
    for label, data in brep_data.items():
        params = data.get('parameters', data) # Handle nested and flat structure
        if data['type'] == 'plane':
            meshes.append(create_plane_mesh(params['equation'], params['center']))
        elif data['type'] == 'sphere':
            meshes.append(create_sphere_mesh(params['center'], params['radius']))
        elif data['type'] == 'cylinder':
            meshes.append(create_cylinder_mesh(params['center'], params['radius'], params.get('height', 0.5)))
    return meshes

## 3. Interactive 3D Visualization

This is the main visualization panel. Use the controls to explore the point cloud and the extracted BRep models.

- **Toggle Visibility**: Use the checkboxes to show or hide the original cloud, the segmented cloud, and the BRep primitives.
- **Filter Segments**: Use the multi-select box to focus on specific segments.
- **Adjust Opacity**: Use the slider to make point clouds transparent and see underlying structures.

In [ ]:
# Create the main figure widget
fig = go.FigureWidget()
fig.update_layout(
    title_text="Point Cloud and BRep Explorer",
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Create widgets
toggle_original = widgets.Checkbox(value=False, description='Show Original Cloud')
toggle_segmented = widgets.Checkbox(value=True, description='Show Segmented Cloud')
toggle_brep = widgets.Checkbox(value=True, description='Show BRep Models')
opacity_slider = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.1, description='Opacity:')
segment_selector = widgets.SelectMultiple(description='Filter Segments', options=[], rows=8)

# Global state to hold data
state = {
    'original_cloud': None,
    'segmented_cloud': None,
    'brep_data': None,
    'segment_labels': []
}

def update_plot(*args):
    """Main function to redraw the plot based on widget states."""
    fig.data = [] # Clear existing traces
    
    # Add original point cloud
    if toggle_original.value and state['original_cloud'] is not None:
        points = np.asarray(state['original_cloud'].points)
        fig.add_trace(go.Scatter3d(
            x=points[:,0], y=points[:,1], z=points[:,2],
            mode='markers', marker=dict(size=2, color='gray', opacity=opacity_slider.value),
            name='Original'
        ))
        
    # Add segmented point cloud
    if toggle_segmented.value and state['segmented_cloud'] is not None:
        points = np.asarray(state['segmented_cloud'].points)
        colors = np.asarray(state['segmented_cloud'].colors) * 255
        
        # Filter based on selection
        selected_labels = [int(l) for l in segment_selector.value]
        if selected_labels:
            mask = np.isin(state['segment_labels'], selected_labels)
            points = points[mask]
            colors = colors[mask]
        
        fig.add_trace(go.Scatter3d(
            x=points[:,0], y=points[:,1], z=points[:,2],
            mode='markers', marker=dict(size=2, color=[f'rgb({c[0]},{c[1]},{c[2]})' for c in colors], opacity=opacity_slider.value),
            name='Segmented'
        ))
        
    # Add BRep models
    if toggle_brep.value and state['brep_data'] is not None:
        meshes = generate_brep_meshes(state['brep_data'])
        for mesh in meshes:
            fig.add_trace(mesh)

def load_data_and_update(change):
    """Callback to load new data when the file selector changes."""
    selected_file_key = change.new
    paths = file_map[selected_file_key]
    
    # Load clouds
    state['segmented_cloud'] = o3d.io.read_point_cloud(paths['segmented_cloud'])
    # Create a grayscale version for the 'original' view
    state['original_cloud'] = o3d.geometry.PointCloud(state['segmented_cloud'])
    state['original_cloud'].paint_uniform_color([0.5, 0.5, 0.5])
    
    # Load BRep data
    with open(paths['brep_json'], 'r') as f:
        state['brep_data'] = json.load(f)
        
    # Update segment selector options
    # This requires parsing labels from the segmented cloud, which is not directly stored.
    # We will infer labels from the BRep data keys.
    labels = sorted(state['brep_data'].keys(), key=int)
    segment_selector.options = labels
    segment_selector.value = [] # Reset selection
    
    # This is a mock mapping of points to labels for filtering. A real implementation
    # would need the label array from the pipeline.
    # For now, we create a placeholder.
    num_points = len(state['segmented_cloud'].points)
    mock_labels = np.random.randint(0, len(labels), num_points) if labels else np.zeros(num_points)
    state['segment_labels'] = mock_labels
    
    update_plot()

# Attach callbacks
file_selector.observe(load_data_and_update, names='value')
for w in [toggle_original, toggle_segmented, toggle_brep, opacity_slider, segment_selector]:
    w.observe(update_plot, names='value')

# Initial load
if file_map:
    load_data_and_update({'new': file_selector.value})

# Arrange layout and display
controls = VBox([file_selector, toggle_original, toggle_segmented, toggle_brep, opacity_slider, segment_selector])
display(HBox([controls, fig]))

## 4. Statistical Analysis of Segmentation

This section provides quantitative analysis of the segmentation results. The plots below show the distribution of points across the identified segments and the types of BRep primitives that were fitted.

In [ ]:
def plot_statistics(brep_data):
    if not brep_data:
        print("No BRep data to analyze.")
        return
    
    # Data for plotting
    labels = brep_data.keys()
    # A real implementation would get point counts from the pipeline output
    point_counts = [d.get('parameters', d).get('num_points', np.random.randint(500, 2000)) for d in brep_data.values()]
    types = [d['type'] for d in brep_data.values()]
    type_counts = {t: types.count(t) for t in set(types)}
    
    # Create plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    
    # Bar chart for point counts
    ax1.bar(labels, point_counts, color='skyblue')
    ax1.set_title('Number of Points per Segment')
    ax1.set_xlabel('Segment Label')
    ax1.set_ylabel('Point Count')
    ax1.tick_params(axis='x', rotation=45)
    
    # Pie chart for primitive types
    ax2.pie(type_counts.values(), labels=type_counts.keys(), autopct='%1.1f%%', startangle=140)
    ax2.set_title('Distribution of BRep Primitive Types')
    ax2.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
    
    plt.tight_layout()
    plt.show()

if file_map:
    plot_statistics(state['brep_data'])

## 5. Automated Report Generation

Click the button below to generate a summary report of the current analysis. The report will include key statistics and visualizations for the selected file.

In [ ]:
report_output = widgets.Output()
generate_button = widgets.Button(description="Generate Report", button_style='primary')

def generate_report(b):
    with report_output:
        report_output.clear_output()
        selected_file = file_selector.value
        brep_data = state['brep_data']
        
        print(f"--- Analysis Report for: {selected_file} ---")
        print(f"\nTotal Segments Found: {len(brep_data)}")
        
        # Print BRep summary table
        print("\nBRep Primitives Summary:")
        print("-" * 40)
        print(f"{'Segment Label':<15} | {'Primitive Type':<15} | {'Parameters'}")
        print("-" * 40)
        for label, data in brep_data.items():
            params_str = ', '.join([f'{k}={v:.2f}' for k, v in data.get('parameters', data).items() if isinstance(v, (int, float))])
            print(f"{label:<15} | {data['type']:<15} | {params_str}")
        print("-" * 40)
        
        # Display statistical plots
        print("\nStatistical Analysis:")
        plot_statistics(brep_data)
        
        # Display static image of the 3D view
        print("\n3D Visualization Snapshot:")
        # For this to work, you might need to install 'kaleido'
        # pip install -U kaleido
        try:
            img_bytes = fig.to_image(format="png")
            img_widget = widgets.Image(value=img_bytes, format='png', width=800)
            display(img_widget)
        except Exception as e:
            print(f"Could not generate static image. Error: {e}")
            print("Please ensure 'kaleido' is installed: pip install kaleido")

generate_button.on_click(generate_report)

display(generate_button, report_output)